# Imbalanced Rare-Defect Classification

Skewed datasets are not a false direction. They are one of the most realistic parts of industrial inspection: most parts are normal, and true failures are rare.

The trap is metric choice. A model can achieve high ordinary accuracy by predicting every item as clean. For rare defects, we should track balanced accuracy, defect recall, false alarm rate, F1, and average precision.

This notebook keeps the QNN direction but changes the evaluation story: a QCNN-style QNN is trained on a balanced subset, then evaluated on a rare-defect test set.

## 1. Imports

The notebook compares classical class weighting, a quantum-kernel SVC, and a QCNN-style QNN.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    f1_score,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM, SVC

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.circuit.library import zz_feature_map
from qiskit.primitives import StatevectorSampler
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit_algorithms.optimizers import COBYLA
from qiskit_machine_learning.algorithms import QSVC
from qiskit_machine_learning.algorithms.classifiers import NeuralNetworkClassifier
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.neural_networks import SamplerQNN

SEED = 8398
np.random.seed(SEED)

## 2. Helpers

The synthetic proxy is an industrial micro-texture task where only the top tail of a coupled texture score is treated as defective.

In [ ]:
def coupled_texture_score(X):
    """Hidden coupled texture score used to build proxy defects."""
    return (
        np.sin(X[:, 0]) * np.cos(X[:, 1])
        + np.sin(X[:, 2] + X[:, 3])
        + 0.25 * np.cos(X[:, 0] - X[:, 2])
    )


def latents_to_microtexture_images(X, size=8, seed=SEED):
    """Render four compact texture factors as small grayscale inspection patches."""
    rng = np.random.default_rng(seed)
    grid = np.linspace(0, 2 * np.pi, size)
    rr, cc = np.meshgrid(grid, grid, indexing="ij")
    images = []

    for x in X:
        img = (
            0.50
            + 0.16 * np.sin(rr + x[0])
            + 0.13 * np.cos(cc + x[1])
            + 0.10 * np.sin(rr + cc + x[2])
            + 0.08 * np.cos(rr - cc + x[3])
        )
        img += rng.normal(0.0, 0.025, size=(size, size))
        images.append(np.clip(img, 0.0, 1.0))

    return np.array(images)


def show_microtextures(images, labels, scores=None, n_show=12, cols=6):
    order = np.arange(min(n_show, len(images)))
    rows = int(np.ceil(len(order) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.5, rows * 1.7))
    axes = np.ravel(axes)

    for ax, idx in zip(axes, order):
        ax.imshow(images[idx], cmap="gray_r", vmin=0, vmax=1)
        label = "DEFECT" if labels[idx] else "clean"
        suffix = "" if scores is None else f"\nscore={scores[idx]:.2f}"
        ax.set_title(f"#{idx} {label}{suffix}", fontsize=8, color="crimson" if labels[idx] else "black")
        ax.set_xticks([])
        ax.set_yticks([])

    for ax in axes[len(order):]:
        ax.axis("off")

    fig.tight_layout()
    plt.show()


def evaluate_predictions(name, y_true, y_pred, scores=None, *, show_confusion=True):
    """Report rare-defect metrics. Label 1 is defect/anomaly."""
    print(f"\n{name}")
    print("ordinary accuracy:", accuracy_score(y_true, y_pred))
    print("balanced accuracy:", balanced_accuracy_score(y_true, y_pred))
    print("defect/anomaly F1:", f1_score(y_true, y_pred, zero_division=0))
    print("defect recall   :", ((y_pred == 1) & (y_true == 1)).sum() / max(1, (y_true == 1).sum()))
    print("false alarm rate:", ((y_pred == 1) & (y_true == 0)).sum() / max(1, (y_true == 0).sum()))
    if scores is not None:
        print("average precision:", average_precision_score(y_true, scores))
        try:
            print("ROC AUC          :", roc_auc_score(y_true, scores))
        except ValueError:
            pass
    print(classification_report(y_true, y_pred, target_names=["clean", "defect"], zero_division=0))

    if show_confusion:
        ConfusionMatrixDisplay.from_predictions(y_true, y_pred, display_labels=["clean", "defect"])
        plt.title(name)
        plt.show()


def evaluate_classifier(name, model, X_eval, y_eval, *, show_confusion=False):
    pred = model.predict(X_eval)
    scores = None
    if hasattr(model, "predict_proba"):
        try:
            scores = model.predict_proba(X_eval)[:, 1]
        except Exception:
            scores = None
    elif hasattr(model, "decision_function"):
        try:
            scores = model.decision_function(X_eval)
        except Exception:
            scores = None
    evaluate_predictions(name, y_eval, pred, scores=scores, show_confusion=show_confusion)
    return pred


def balanced_subset(X_data, y_data, n_clean_per_defect=3, seed=SEED):
    """Use every rare defect and a controlled number of clean examples."""
    rng = np.random.default_rng(seed)
    defect_idx = np.flatnonzero(y_data == 1)
    clean_idx = np.flatnonzero(y_data == 0)
    n_clean = min(len(clean_idx), max(1, len(defect_idx) * n_clean_per_defect))
    chosen = defect_idx.tolist() + rng.choice(clean_idx, size=n_clean, replace=False).tolist()
    rng.shuffle(chosen)
    return X_data[chosen], y_data[chosen]


def build_qcnn_style_circuit(n_qubits=4):
    """Compact QCNN-style QNN for four texture features."""
    x = ParameterVector("x", n_qubits)
    theta = ParameterVector("theta", 10)
    qc = QuantumCircuit(n_qubits)

    for q in range(n_qubits):
        qc.ry(x[q], q)

    qc.cx(0, 1)
    qc.ry(theta[0], 1)
    qc.rz(theta[1], 0)
    qc.cx(1, 0)
    qc.ry(theta[2], 0)

    qc.cx(2, 3)
    qc.ry(theta[0], 3)
    qc.rz(theta[1], 2)
    qc.cx(3, 2)
    qc.ry(theta[2], 2)

    qc.cx(1, 0)
    qc.ry(theta[3], 0)
    qc.cz(1, 0)
    qc.cx(3, 2)
    qc.ry(theta[4], 2)
    qc.cz(3, 2)

    qc.cx(0, 2)
    qc.ry(theta[5], 2)
    qc.rz(theta[6], 0)
    qc.cx(2, 0)
    qc.ry(theta[7], 0)
    qc.ry(theta[8], 2)
    qc.cz(0, 2)
    qc.ry(theta[9], 0)

    return qc, list(x), list(theta)


def readout_q0(measured_integer):
    return measured_integer & 1

## 3. Generate a Rare-Defect Dataset

`DEFECT_RATE=0.08` means only about 8 percent of samples are defective. That is still less skewed than many real inspection settings, but enough to expose the accuracy trap.

In [ ]:
N_SAMPLES = 240
DEFECT_RATE = 0.08

rng = np.random.default_rng(SEED + 500)
X = rng.uniform(0.0, 2 * np.pi, size=(N_SAMPLES, 4))
scores = coupled_texture_score(X)
threshold = np.quantile(scores, 1 - DEFECT_RATE)
y = (scores >= threshold).astype(int)
images = latents_to_microtexture_images(X, size=8, seed=SEED)

print("feature shape:", X.shape)
print("class balance:", {"clean": int((y == 0).sum()), "defect": int((y == 1).sum())})
print("defect rate:", y.mean())
show_microtextures(images, y, scores=scores, n_show=12, cols=6)

## 4. Split Data

The split is stratified so the rare defect class appears in both train and test.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.35,
    random_state=SEED,
    stratify=y,
)

print("train:", X_train.shape, "clean:", int((y_train == 0).sum()), "defect:", int((y_train == 1).sum()))
print("test :", X_test.shape, "clean:", int((y_test == 0).sum()), "defect:", int((y_test == 1).sum()))

## 5. The Accuracy Trap

Predicting every sample as clean looks good under ordinary accuracy, but it has zero defect recall.

In [ ]:
all_clean = np.zeros_like(y_test)
evaluate_predictions("All-clean baseline", y_test, all_clean, show_confusion=False)

## 6. Classical Baselines With and Without Class Weighting

Class weighting is a serious baseline. If a quantum method cannot beat or complement this, the final story should be cautious.

In [ ]:
classical_models = {
    "Linear SVM, unweighted": Pipeline([
        ("scale", StandardScaler()),
        ("model", SVC(kernel="linear", random_state=SEED)),
    ]),
    "Linear SVM, class_weight=balanced": Pipeline([
        ("scale", StandardScaler()),
        ("model", SVC(kernel="linear", class_weight="balanced", random_state=SEED)),
    ]),
    "RBF SVM, unweighted": Pipeline([
        ("scale", StandardScaler()),
        ("model", SVC(kernel="rbf", random_state=SEED)),
    ]),
    "RBF SVM, class_weight=balanced": Pipeline([
        ("scale", StandardScaler()),
        ("model", SVC(kernel="rbf", class_weight="balanced", probability=True, random_state=SEED)),
    ]),
    "Random Forest, class_weight=balanced": RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=SEED,
    ),
}

for name, model in classical_models.items():
    model.fit(X_train, y_train)
    evaluate_classifier(name, model, X_test, y_test, show_confusion=False)

## 7. Quantum Kernel SVC on a Balanced Training Subset

This is a conservative quantum fallback. The subset keeps all rare defects and a limited number of clean examples so the SVC sees both classes.

In [ ]:
X_kernel_train, y_kernel_train = balanced_subset(
    X_train,
    y_train,
    n_clean_per_defect=3,
    seed=SEED,
)

feature_map = zz_feature_map(feature_dimension=4, reps=1)
quantum_kernel = FidelityQuantumKernel(feature_map=feature_map)
qsvc = QSVC(quantum_kernel=quantum_kernel, class_weight="balanced")
qsvc.fit(X_kernel_train, y_kernel_train)
evaluate_classifier("Quantum Kernel QSVC", qsvc, X_test, y_test, show_confusion=False)

## 8. QCNN-Style QNN on a Balanced Training Subset

The QNN is not trained on the full skewed set. It is trained on every rare defect plus a controlled number of clean examples, then evaluated on the true imbalanced test distribution.

In [ ]:
QNN_MAXITER = 50

X_qnn_train, y_qnn_train = balanced_subset(
    X_train,
    y_train,
    n_clean_per_defect=3,
    seed=SEED,
)

qcnn_circuit, input_params, weight_params = build_qcnn_style_circuit(n_qubits=4)
sampler_qnn = SamplerQNN(
    circuit=qcnn_circuit,
    sampler=StatevectorSampler(seed=SEED),
    input_params=input_params,
    weight_params=weight_params,
    interpret=readout_q0,
    output_shape=2,
)

rng = np.random.default_rng(SEED)
qcnn_classifier = NeuralNetworkClassifier(
    neural_network=sampler_qnn,
    optimizer=COBYLA(maxiter=QNN_MAXITER),
    initial_point=rng.uniform(-0.2, 0.2, size=len(weight_params)),
)

qcnn_classifier.fit(X_qnn_train, y_qnn_train)
qcnn_pred = qcnn_classifier.predict(X_test)
qcnn_scores = qcnn_classifier.predict_proba(X_test)[:, 1]
evaluate_predictions("QCNN-style QNN", y_test, qcnn_pred, scores=qcnn_scores)

## 9. Precision-Recall View

For rare defects, the precision-recall curve is often more useful than a single accuracy number. A high-recall detector may be useful if the false alarms can be sent to a second inspection stage.

In [ ]:
precision, recall, pr_thresholds = precision_recall_curve(y_test, qcnn_scores)

plt.figure(figsize=(5, 4))
plt.plot(recall, precision, marker=".")
plt.xlabel("defect recall")
plt.ylabel("precision")
plt.title("QCNN-style QNN precision-recall curve")
plt.grid(True, alpha=0.3)
plt.show()

print("QCNN average precision:", average_precision_score(y_test, qcnn_scores))

## 10. Takeaway

Skewed classification is viable and real. It is not a side quest. The honest claim is:

> Quantum/QNN models may be useful as high-recall rare-defect screeners when labeled failures are scarce, but they must be judged with rare-event metrics and compared against class-weighted classical baselines.

## References and AI Tools Disclosure

This notebook was drafted with OpenAI Codex/GPT-5 assistance in the local hackathon workspace. The team should treat the code and results as AI-assisted prototypes and verify claims before presenting them.

AI-assisted notebooks in this `main_challenge` folder:

- `01_defect_classical_hardness_audit.ipynb`
- `02_quantum_kernel_teacher_defects.ipynb`
- `03_projected_quantum_kernel_features.ipynb`
- `04_qnn_vs_kernel_bakeoff.ipynb`
- `05_qcnn_industrial_microdefects.ipynb`
- `06_data_reuploading_qnn_microdefects.ipynb`
- `07_qnn_kernel_pivot_scoreboard.ipynb`
- `08_imbalanced_rare_defect_qnn.ipynb`
- `09_normal_only_anomaly_detection.ipynb`

References used for the quantum-ML direction:

- Cong, Choi, and Lukin, "Quantum convolutional neural networks," Nature Physics 15, 1273-1278 (2019): https://www.nature.com/articles/s41567-019-0648-8
- Pérez-Salinas et al., "Data re-uploading for a universal quantum classifier," Quantum 4, 226 (2020): https://doi.org/10.22331/q-2020-02-06-226 and https://arxiv.org/abs/1907.02085
- McClean et al., "Barren plateaus in quantum neural network training landscapes," Nature Communications 9, 4812 (2018): https://www.nature.com/articles/s41467-018-07090-4
- Havlíček et al., "Supervised learning with quantum-enhanced feature spaces," Nature 567, 209-212 (2019): https://www.nature.com/articles/s41586-019-0980-2
- Schölkopf et al., "Estimating the Support of a High-Dimensional Distribution," Neural Computation 13(7), 1443-1471 (2001): https://doi.org/10.1162/089976601750264965
- Ngairangbam, Spannowsky, and Takeuchi, "Anomaly detection in high-energy physics using a quantum autoencoder," Physical Review D 105, 095004 (2022): https://arxiv.org/abs/2112.04958
- Quantum Patch-Based Autoencoder for Anomaly Segmentation: https://arxiv.org/abs/2404.17613
- Quantum Machine Learning Algorithms for Anomaly Detection: a Review: https://arxiv.org/abs/2408.11047
- Qiskit Machine Learning `SamplerQNN` documentation: https://qiskit-community.github.io/qiskit-machine-learning/stubs/qiskit_machine_learning.neural_networks.SamplerQNN.html